# 02 - True GAN Face Crop And Inversion

This is the first genuinely DragGAN-compatible step. It does **not** do geometric warping. It crops the replacement dog's face and projects that real face into the AFHQ Dog StyleGAN2-ADA latent space.

The risk is real: GAN inversion may drift in identity/color because the generator can only reconstruct images that live near its training distribution. We keep the reconstruction and diagnostics visible so bad inversions are obvious before continuing.

In [ ]:
%matplotlib inline

import copy
import json
import math
import os
import pickle
import subprocess
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
PIPELINE_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline"
CUTOUT_ROOT = PIPELINE_ROOT / "01_cutout"
DRAGGAN_ROOT = PROJECT_ROOT / "external" / "DragGAN"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "draggan"
AFHQ_DOG_PKL = CHECKPOINT_DIR / "stylegan2-afhqdog-512x512.pkl"
AFHQ_DOG_URL = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/afhqdog.pkl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("True DragGAN/StyleGAN inversion needs CUDA for practical runtime.")

print("Project root:", PROJECT_ROOT)
print("Pipeline root:", PIPELINE_ROOT)
print("DragGAN repo:", DRAGGAN_ROOT)
print("AFHQ Dog checkpoint:", AFHQ_DOG_PKL)
print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:
MAX_PAIRS = int(os.environ.get("DOG_TRUE_DRAGGAN_MAX_PAIRS", "1"))
INVERSION_STEPS = int(os.environ.get("DOG_TRUE_DRAGGAN_INVERSION_STEPS", "600"))
INVERSION_LR = float(os.environ.get("DOG_TRUE_DRAGGAN_INVERSION_LR", "0.03"))
FACE_CROP_SCALE = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_CROP_SCALE", "1.30"))
USE_LPIPS_IF_AVAILABLE = True

# Stronger inversion: W+ is optimized with pixel loss, LPIPS, light W regularization,
# and optional StyleGAN noise-buffer optimization. This usually preserves sharper facial texture.
INVERSION_PIXEL_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_PIXEL_WEIGHT", "1.0"))
INVERSION_LPIPS_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_LPIPS_WEIGHT", "0.5"))
INVERSION_W_REG_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_W_REG_WEIGHT", "0.001"))
OPTIMIZE_NOISE = os.environ.get("DOG_TRUE_DRAGGAN_OPTIMIZE_NOISE", "True").lower() in {"1", "true", "yes", "on"}
NOISE_REG_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_NOISE_REG_WEIGHT", "100000.0"))

# PTI (Pivotal Tuning Inversion): after finding W+, lightly tune this image's generator copy
# so reconstruction is closer before DragGAN starts moving handles.
ENABLE_PTI = os.environ.get("DOG_TRUE_DRAGGAN_ENABLE_PTI", "True").lower() in {"1", "true", "yes", "on"}
PTI_STEPS = int(os.environ.get("DOG_TRUE_DRAGGAN_PTI_STEPS", "600"))
PTI_LR = float(os.environ.get("DOG_TRUE_DRAGGAN_PTI_LR", "0.0002"))
PTI_PIXEL_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_PTI_PIXEL_WEIGHT", "1.0"))
PTI_LPIPS_WEIGHT = float(os.environ.get("DOG_TRUE_DRAGGAN_PTI_LPIPS_WEIGHT", "0.5"))

OUTPUT_ROOT = PIPELINE_ROOT / "02_true_draggan_face_inversion"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Max pairs:", MAX_PAIRS)
print("Inversion steps:", INVERSION_STEPS)
print("Inversion LR:", INVERSION_LR)
print("Face crop scale:", FACE_CROP_SCALE)
print("LPIPS weight:", INVERSION_LPIPS_WEIGHT)
print("Optimize StyleGAN noise buffers:", OPTIMIZE_NOISE)
print("PTI enabled:", ENABLE_PTI)
print("PTI steps:", PTI_STEPS)
print("PTI LR:", PTI_LR)
print("Output root:", OUTPUT_ROOT)


In [ ]:

def ensure_draggan_repo():
    if DRAGGAN_ROOT.exists():
        return
    DRAGGAN_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/XingangPan/DragGAN.git",
        str(DRAGGAN_ROOT),
    ])


def ensure_afhq_dog_checkpoint():
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    if AFHQ_DOG_PKL.exists():
        return
    import requests
    from tqdm.auto import tqdm

    tmp = AFHQ_DOG_PKL.with_suffix(".part")
    with requests.get(AFHQ_DOG_URL, stream=True, timeout=30) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc="Downloading AFHQ Dog StyleGAN2-ADA") as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    tmp.replace(AFHQ_DOG_PKL)


def add_draggan_to_path():
    ensure_draggan_repo()
    draggan_str = str(DRAGGAN_ROOT)
    if draggan_str not in sys.path:
        sys.path.insert(0, draggan_str)



def force_stylegan_reference_ops():
    """Force StyleGAN/DragGAN to use PyTorch reference ops instead of compiling CUDA plugins.

    The official StyleGAN code tries to build custom CUDA extensions such as
    `bias_act_plugin` and `upfirdn2d_plugin`. On Windows this requires MSVC
    Build Tools. For this notebook branch we prefer reliability over speed, so
    these wrappers route calls to the built-in reference implementations.
    """
    from torch_utils.ops import bias_act, filtered_lrelu, upfirdn2d

    def bias_act_ref(x, b=None, dim=1, act="linear", alpha=None, gain=None, clamp=None, impl="ref"):
        return bias_act._bias_act_ref(x=x, b=b, dim=dim, act=act, alpha=alpha, gain=gain, clamp=clamp)

    def upfirdn2d_ref(x, f, up=1, down=1, padding=0, flip_filter=False, gain=1, impl="ref"):
        return upfirdn2d._upfirdn2d_ref(x=x, f=f, up=up, down=down, padding=padding, flip_filter=flip_filter, gain=gain)

    def filtered_lrelu_ref(x, fu=None, fd=None, b=None, up=1, down=1, padding=0, gain=np.sqrt(2), slope=0.2, clamp=None, flip_filter=False, impl="ref"):
        return filtered_lrelu._filtered_lrelu_ref(
            x=x,
            fu=fu,
            fd=fd,
            b=b,
            up=up,
            down=down,
            padding=padding,
            gain=gain,
            slope=slope,
            clamp=clamp,
            flip_filter=flip_filter,
        )

    bias_act.bias_act = bias_act_ref
    upfirdn2d.upfirdn2d = upfirdn2d_ref
    filtered_lrelu.filtered_lrelu = filtered_lrelu_ref
    print("StyleGAN custom CUDA plugins disabled: using PyTorch reference ops.")


def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def read_rgba(path):
    return np.array(Image.open(path).convert("RGBA"))


def read_alpha_float(path):
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32) / 255.0


def apply_soft_alpha_to_rgba(rgba, alpha):
    alpha = np.clip(alpha.astype(np.float32), 0.0, 1.0)
    if alpha.shape != rgba.shape[:2]:
        alpha = cv2.resize(alpha, (rgba.shape[1], rgba.shape[0]), interpolation=cv2.INTER_LINEAR)
    out = rgba.copy()
    out[..., 3] = np.uint8(alpha * 255)
    out[out[..., 3] == 0, :3] = 0
    return out


def save_rgb(path, image):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.clip(image, 0, 255).astype(np.uint8)).save(path)


def show_images(items, cols=3, figsize=(16, 8)):
    rows = math.ceil(len(items) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, (title, image) in zip(axes, items):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def load_active_cutout_jobs(max_pairs):
    manifest = CUTOUT_ROOT / "_active_batch_manifest.json"
    if not manifest.exists():
        manifest = CUTOUT_ROOT / "batch_manifest.json"
    if not manifest.exists():
        raise FileNotFoundError(f"No cutout manifest found at {CUTOUT_ROOT}. Run notebook 01 first.")
    with open(manifest, "r", encoding="utf-8") as f:
        jobs = json.load(f)
    if max_pairs is not None:
        jobs = jobs[: int(max_pairs)]
    return jobs


DOG_KEYPOINT_NAMES = [
    "front_left_paw", "front_left_knee", "front_left_elbow",
    "rear_left_paw", "rear_left_knee", "rear_left_elbow",
    "front_right_paw", "front_right_knee", "front_right_elbow",
    "rear_right_paw", "rear_right_knee", "rear_right_elbow",
    "tail_start", "tail_end",
    "left_ear_base", "right_ear_base",
    "nose", "chin",
    "left_ear_tip", "right_ear_tip",
    "left_eye", "right_eye",
    "withers", "throat",
]
FACE_NAMES = ["nose", "chin", "left_eye", "right_eye", "left_ear_base", "right_ear_base", "left_ear_tip", "right_ear_tip", "throat"]


KEYPOINT_CONF = 0.20


def deserialize_pose_record(record):
    if record is None:
        return None
    out = dict(record)
    out["points"] = np.asarray(record["points"], dtype=np.float32)
    if "visible" in record:
        out["visible"] = np.asarray(record["visible"], dtype=bool)
    elif "confs" in record:
        out["confs"] = np.asarray(record["confs"], dtype=np.float32)
        out["visible"] = out["confs"] >= KEYPOINT_CONF
    else:
        # Older metadata always stored points, but not an explicit visibility mask.
        # Treat finite non-zero points as visible so the notebook can still run.
        pts = out["points"]
        out["visible"] = np.isfinite(pts).all(axis=1) & (np.linalg.norm(pts, axis=1) > 1e-3)
    return out


def face_keypoints(pose):
    points = {}
    for name in FACE_NAMES:
        idx = DOG_KEYPOINT_NAMES.index(name)
        if idx < len(pose["points"]) and bool(pose["visible"][idx]):
            points[name] = pose["points"][idx].astype(np.float32)
    return points


def stable_face_crop_points(points, image_shape, min_side=96):
    """Return robust face-crop anchors without ear-tip/throat outliers.

    Dog pose ear-tip and throat points are useful for pose reasoning, but they are
    too unstable for a tight face crop. A single bad ear-tip can pull the crop far
    away from the muzzle. For DragGAN inversion we keep the crop driven by nose,
    chin, eyes, and ear bases; when only nose/chin are available, synthetic cheek
    anchors keep the square crop centered on the face instead of stretching to an
    unrelated outlier.
    """
    h, w = image_shape[:2]
    stable_names = ["nose", "chin", "left_eye", "right_eye", "left_ear_base", "right_ear_base"]
    anchors = [np.asarray(points[name], dtype=np.float32) for name in stable_names if name in points]
    nose = np.asarray(points["nose"], dtype=np.float32) if "nose" in points else None
    chin = np.asarray(points["chin"], dtype=np.float32) if "chin" in points else None

    if nose is not None:
        anchors.append(nose)
        if chin is not None:
            face_len = max(float(np.linalg.norm(chin - nose)), float(min_side) * 0.45)
        else:
            face_len = float(min_side) * 0.55
        # Synthetic side anchors prevent a nose/chin-only crop from becoming a
        # narrow vertical strip and avoid using noisy ear-tip points.
        anchors.extend([
            nose + np.asarray([-face_len * 0.70, face_len * 0.15], dtype=np.float32),
            nose + np.asarray([face_len * 0.70, face_len * 0.15], dtype=np.float32),
            nose + np.asarray([0.0, -face_len * 0.55], dtype=np.float32),
        ])
        if chin is not None:
            anchors.append(chin)

    if len(anchors) < 2:
        raise RuntimeError("Too few stable face keypoints for true DragGAN face crop.")

    pts = np.asarray(anchors, dtype=np.float32)
    pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
    pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)
    return pts


def square_crop_box_from_points(points, image_shape, scale=1.80, min_side=96):
    h, w = image_shape[:2]
    pts = stable_face_crop_points(points, image_shape, min_side=min_side)
    x1, y1 = pts.min(axis=0)
    x2, y2 = pts.max(axis=0)
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    side = max(float(x2 - x1), float(y2 - y1), float(min_side)) * float(scale)
    x1 = int(round(cx - side / 2.0))
    y1 = int(round(cy - side / 2.0))
    x2 = int(round(cx + side / 2.0))
    y2 = int(round(cy + side / 2.0))
    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - w)
    pad_bottom = max(0, y2 - h)
    x1c, y1c = max(0, x1), max(0, y1)
    x2c, y2c = min(w, x2), min(h, y2)
    return [x1, y1, x2, y2], [x1c, y1c, x2c, y2c], [pad_left, pad_top, pad_right, pad_bottom]


def crop_square_with_padding(image, raw_box, clipped_box, padding, fill=(127, 127, 127, 0)):
    x1, y1, x2, y2 = raw_box
    x1c, y1c, x2c, y2c = clipped_box
    side = max(1, x2 - x1, y2 - y1)
    channels = image.shape[2] if image.ndim == 3 else 1
    if channels == 4:
        canvas = np.zeros((side, side, 4), dtype=np.uint8)
        canvas[..., :] = np.array(fill, dtype=np.uint8)
    else:
        canvas = np.zeros((side, side, channels), dtype=np.uint8)
        canvas[..., :] = np.array(fill[:channels], dtype=np.uint8)
    dx = x1c - x1
    dy = y1c - y1
    canvas[dy:dy + (y2c - y1c), dx:dx + (x2c - x1c)] = image[y1c:y2c, x1c:x2c]
    return canvas


def points_to_crop512(points, raw_box):
    x1, y1, x2, y2 = raw_box
    side = max(1, x2 - x1, y2 - y1)
    out = {}
    for name, pt in points.items():
        out[name] = [
            float((pt[0] - x1) / side * 512.0),
            float((pt[1] - y1) / side * 512.0),
        ]
    return out


def draw_points(image, points, color=(255, 60, 60)):
    out = image.copy()
    for name, (x, y) in points.items():
        cv2.circle(out, (int(round(x)), int(round(y))), 5, color, -1)
        cv2.putText(out, name, (int(round(x)) + 6, int(round(y)) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.38, color, 1, cv2.LINE_AA)
    return out


In [ ]:
ensure_draggan_repo()
ensure_afhq_dog_checkpoint()
add_draggan_to_path()
force_stylegan_reference_ops()

import dnnlib
import legacy

with dnnlib.util.open_url(str(AFHQ_DOG_PKL), verbose=True) as f:
    BASE_NETWORK_DATA = legacy.load_network_pkl(f)

G = BASE_NETWORK_DATA["G_ema"].to(DEVICE).eval().requires_grad_(False)

print("Loaded generator resolution:", G.img_resolution)
print("Number of W+ slots:", G.num_ws)


In [ ]:
lpips_model = None
if USE_LPIPS_IF_AVAILABLE:
    try:
        import lpips
        lpips_model = lpips.LPIPS(net="vgg").to(DEVICE).eval()
        print("LPIPS enabled.")
    except Exception as exc:
        print("LPIPS unavailable; using pixel reconstruction loss only.")
        print(type(exc).__name__, exc)

In [ ]:
def image_to_tensor_512(rgb):
    image = Image.fromarray(rgb.astype(np.uint8)).resize((512, 512), Image.Resampling.LANCZOS)
    arr = np.array(image).astype(np.float32) / 127.5 - 1.0
    return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(DEVICE)


def tensor_to_image(tensor):
    arr = (tensor.detach().clamp(-1, 1)[0].permute(1, 2, 0).cpu().numpy() * 127.5 + 127.5)
    return np.clip(arr, 0, 255).astype(np.uint8)


@torch.no_grad()
def compute_w_avg(G, samples=10000, seed=7):
    rng = np.random.RandomState(seed)
    z = torch.from_numpy(rng.randn(samples, G.z_dim)).to(DEVICE, dtype=torch.float32)
    c = torch.zeros([samples, G.c_dim], device=DEVICE)
    w = G.mapping(z, c, truncation_psi=1.0)
    if w.ndim == 2:
        w = w.unsqueeze(1)
    return w.mean(0, keepdim=True)


def normalize_w_for_synthesis(G, w):
    if w.ndim == 2:
        w = w.unsqueeze(1)
    if w.shape[1] == G.num_ws:
        return w
    if w.shape[1] > G.num_ws and w.shape[1] % G.num_ws == 0:
        print(f"Warning: latent had {w.shape[1]} W+ slots, but this generator expects {G.num_ws}; using the first {G.num_ws} slots.")
        return w[:, : G.num_ws, :]
    if w.shape[1] == 1:
        return w.repeat(1, G.num_ws, 1)
    raise RuntimeError(f"Cannot adapt latent shape {tuple(w.shape)} to G.num_ws={G.num_ws}.")


def synthesize_from_w(G, w):
    w = normalize_w_for_synthesis(G, w)
    return G.synthesis(w, noise_mode="const")


def collect_noise_buffers(G):
    return {name: buf for name, buf in G.synthesis.named_buffers() if "noise_const" in name}


@torch.no_grad()
def normalize_noise_buffers(noise_bufs):
    for buf in noise_bufs.values():
        buf -= buf.mean()
        buf *= torch.rsqrt(buf.square().mean() + 1e-8)


def noise_regularize(noise_bufs):
    reg = torch.zeros([], device=DEVICE)
    for noise in noise_bufs.values():
        n = noise
        if n.ndim == 2:
            n = n[None, None, :, :]
        elif n.ndim == 3:
            n = n[None, :, :, :]
        while True:
            reg = reg + (n * torch.roll(n, shifts=1, dims=3)).mean().pow(2)
            reg = reg + (n * torch.roll(n, shifts=1, dims=2)).mean().pow(2)
            if n.shape[2] <= 8 or n.shape[3] <= 8:
                break
            n = F.avg_pool2d(n, kernel_size=2)
    return reg


def reconstruction_loss_terms(synth, target, pixel_weight, lpips_weight):
    pixel_loss = F.mse_loss(synth, target)
    lpips_loss = torch.zeros([], device=DEVICE)
    if lpips_model is not None and lpips_weight > 0:
        lpips_loss = lpips_model(synth, target).mean()
    total = pixel_weight * pixel_loss + lpips_weight * lpips_loss
    return total, pixel_loss, lpips_loss


def project_face_to_wplus(G, target_rgb, steps=900, lr=0.04):
    target = image_to_tensor_512(target_rgb)
    w_avg = compute_w_avg(G, samples=4096)
    print(f"Generator latent layout: z_dim={G.z_dim}, num_ws={G.num_ws}, mapped_w_shape={tuple(w_avg.shape)}")

    w_opt = normalize_w_for_synthesis(G, w_avg).detach().clone()
    w_anchor = w_opt.detach().clone()
    w_opt.requires_grad_(True)

    noise_bufs = collect_noise_buffers(G)
    params = [w_opt]
    if OPTIMIZE_NOISE and noise_bufs:
        for buf in noise_bufs.values():
            buf.requires_grad = True
            params.append(buf)
        print("Optimizing StyleGAN noise buffers:", len(noise_bufs))
    else:
        print("StyleGAN noise-buffer optimization disabled.")

    optimizer = torch.optim.Adam(params, lr=lr)
    history = []

    for step in range(steps):
        synth = synthesize_from_w(G, w_opt)
        recon_loss, pixel_loss, lpips_loss = reconstruction_loss_terms(
            synth,
            target,
            INVERSION_PIXEL_WEIGHT,
            INVERSION_LPIPS_WEIGHT,
        )
        w_reg_loss = (w_opt - w_anchor).square().mean()
        noise_reg_loss = noise_regularize(noise_bufs) if OPTIMIZE_NOISE and noise_bufs else torch.zeros([], device=DEVICE)
        loss = recon_loss + INVERSION_W_REG_WEIGHT * w_reg_loss + NOISE_REG_WEIGHT * noise_reg_loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if OPTIMIZE_NOISE and noise_bufs:
            normalize_noise_buffers(noise_bufs)

        if step % 50 == 0 or step == steps - 1:
            record = {
                "step": int(step),
                "loss": float(loss.detach().cpu()),
                "pixel_loss": float(pixel_loss.detach().cpu()),
                "lpips_loss": float(lpips_loss.detach().cpu()),
                "w_reg_loss": float(w_reg_loss.detach().cpu()),
                "noise_reg_loss": float(noise_reg_loss.detach().cpu()),
            }
            history.append(record)
            print(
                f"step {step:04d} | loss={record['loss']:.5f} | pixel={record['pixel_loss']:.5f} "
                f"| lpips={record['lpips_loss']:.5f} | w_reg={record['w_reg_loss']:.6f}"
            )

    final = tensor_to_image(synthesize_from_w(G, w_opt))
    for buf in noise_bufs.values():
        buf.requires_grad = False
    return w_opt.detach().cpu().numpy(), final, history


def run_pti_tuning(G_pair, target_rgb, w_np):
    if not ENABLE_PTI or PTI_STEPS <= 0:
        w = torch.from_numpy(w_np).to(DEVICE, dtype=torch.float32)
        return G_pair, tensor_to_image(synthesize_from_w(G_pair, w)), []

    target = image_to_tensor_512(target_rgb)
    w = torch.from_numpy(w_np).to(DEVICE, dtype=torch.float32).detach()

    G_pair.train()
    G_pair.requires_grad_(False)
    G_pair.synthesis.requires_grad_(True)
    optimizer = torch.optim.Adam(G_pair.synthesis.parameters(), lr=PTI_LR)
    history = []

    print(f"PTI tuning generator copy for {PTI_STEPS} steps.")
    for step in range(PTI_STEPS):
        synth = synthesize_from_w(G_pair, w)
        loss, pixel_loss, lpips_loss = reconstruction_loss_terms(
            synth,
            target,
            PTI_PIXEL_WEIGHT,
            PTI_LPIPS_WEIGHT,
        )
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if step % 50 == 0 or step == PTI_STEPS - 1:
            record = {
                "step": int(step),
                "loss": float(loss.detach().cpu()),
                "pixel_loss": float(pixel_loss.detach().cpu()),
                "lpips_loss": float(lpips_loss.detach().cpu()),
            }
            history.append(record)
            print(
                f"pti {step:04d} | loss={record['loss']:.5f} "
                f"| pixel={record['pixel_loss']:.5f} | lpips={record['lpips_loss']:.5f}"
            )

    G_pair.eval().requires_grad_(False)
    final = tensor_to_image(synthesize_from_w(G_pair, w))
    return G_pair, final, history


def save_generator_for_draggan(G_tuned, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    data = dict(BASE_NETWORK_DATA)
    G_save = copy.deepcopy(G_tuned).eval().requires_grad_(False).cpu()
    data["G_ema"] = G_save
    data["G"] = copy.deepcopy(G_save).eval().requires_grad_(False)
    with open(path, "wb") as f:
        pickle.dump(data, f)
    return path


def estimate_dark_blob_center(gray, x0, y0, x1, y1, prefer_xy, percentile=18):
    h, w = gray.shape[:2]
    xa, xb = int(round(x0 * w)), int(round(x1 * w))
    ya, yb = int(round(y0 * h)), int(round(y1 * h))
    xa, xb = max(0, xa), min(w, xb)
    ya, yb = max(0, ya), min(h, yb)
    if xb <= xa or yb <= ya:
        return None
    roi = gray[ya:yb, xa:xb]
    threshold = np.percentile(roi, percentile)
    mask = np.uint8(roi <= threshold) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return None
    prefer = np.asarray([prefer_xy[0] * w, prefer_xy[1] * h], dtype=np.float32)
    best = None
    best_score = -1.0
    for label in range(1, num_labels):
        area = float(stats[label, cv2.CC_STAT_AREA])
        if area < 8:
            continue
        cx, cy = centroids[label]
        pt = np.asarray([xa + cx, ya + cy], dtype=np.float32)
        dist = float(np.linalg.norm(pt - prefer))
        score = area / (1.0 + dist)
        if score > best_score:
            best_score = score
            best = pt
    return best


def estimate_inversion_face_points(face_rgb):
    """Estimate display/DragGAN handles from the GAN-projected face image, not YOLO pose.

    AFHQ Dog inversion produces an aligned 512x512 dog-face image. For this branch we
    only need stable approximate handles; exact breed-specific anatomy is less important
    than keeping all handles in the same GAN coordinate system.
    """
    gray = cv2.cvtColor(face_rgb, cv2.COLOR_RGB2GRAY)
    points = {}
    diagnostics = {"source": "gan_projection_dark_blob_heuristic"}

    nose = estimate_dark_blob_center(gray, 0.30, 0.38, 0.70, 0.78, (0.50, 0.58), percentile=16)
    left_eye = estimate_dark_blob_center(gray, 0.15, 0.16, 0.50, 0.50, (0.34, 0.33), percentile=14)
    right_eye = estimate_dark_blob_center(gray, 0.50, 0.16, 0.85, 0.50, (0.66, 0.33), percentile=14)

    if nose is None:
        # Keep the branch notebook-stable without falling back to YOLO. AFHQ dog
        # faces are roughly centered after inversion, so this canonical point is
        # a safer failure mode than importing noisy pose metadata back in.
        nose = np.asarray([face_rgb.shape[1] * 0.50, face_rgb.shape[0] * 0.58], dtype=np.float32)
        diagnostics["nose_fallback"] = "canonical_afhq_center"
    else:
        diagnostics["nose_fallback"] = "none"

    points["nose"] = nose.astype(np.float32)
    chin = nose + np.asarray([0.0, 58.0], dtype=np.float32)
    chin[0] = np.clip(chin[0], 0, face_rgb.shape[1] - 1)
    chin[1] = np.clip(chin[1], 0, face_rgb.shape[0] - 1)
    points["chin"] = chin
    if left_eye is not None:
        points["left_eye"] = left_eye.astype(np.float32)
    if right_eye is not None:
        points["right_eye"] = right_eye.astype(np.float32)

    if left_eye is not None and nose is not None:
        left_ear_base = left_eye + 0.70 * (left_eye - nose)
        left_ear_base[0] = np.clip(left_ear_base[0], 0, face_rgb.shape[1] - 1)
        left_ear_base[1] = np.clip(left_ear_base[1], 0, face_rgb.shape[0] - 1)
        points["left_ear_base"] = left_ear_base.astype(np.float32)
    if right_eye is not None and nose is not None:
        right_ear_base = right_eye + 0.70 * (right_eye - nose)
        right_ear_base[0] = np.clip(right_ear_base[0], 0, face_rgb.shape[1] - 1)
        right_ear_base[1] = np.clip(right_ear_base[1], 0, face_rgb.shape[0] - 1)
        points["right_ear_base"] = right_ear_base.astype(np.float32)

    diagnostics["detected_names"] = sorted(points.keys())
    diagnostics["has_driver_nose"] = "nose" in points
    return {name: value.tolist() for name, value in points.items()}, diagnostics



In [ ]:
jobs = load_active_cutout_jobs(MAX_PAIRS)
manifest = []

for idx, item in enumerate(jobs, start=1):
    pair_name = item["pair_name"]
    metadata_path = PROJECT_ROOT / item["metadata_path"]
    with open(metadata_path, "r", encoding="utf-8") as f:
        cutout_meta = json.load(f)

    output_dir = OUTPUT_ROOT / pair_name
    output_dir.mkdir(parents=True, exist_ok=True)
    rep_rgba = read_rgba(PROJECT_ROOT / cutout_meta["replacement"]["rgba_path"])
    replacement_alpha_source = "rgba_embedded_alpha"
    soft_alpha_rel = cutout_meta["replacement"].get("soft_alpha_path")
    if soft_alpha_rel:
        soft_alpha_path = PROJECT_ROOT / soft_alpha_rel
        if soft_alpha_path.exists():
            rep_rgba = apply_soft_alpha_to_rgba(rep_rgba, read_alpha_float(soft_alpha_path))
            replacement_alpha_source = "soft_alpha_path"
    org_pose = deserialize_pose_record(cutout_meta["original"].get("cutout_pose"))
    rep_pose = deserialize_pose_record(cutout_meta["replacement"].get("cutout_pose"))
    if rep_pose is None or org_pose is None:
        raise RuntimeError(f"Missing cutout pose metadata for {pair_name}. Re-run notebook 01.")

    rep_face_points_pose = face_keypoints(rep_pose)
    org_face_points_pose = face_keypoints(org_pose)
    raw_box, clipped_box, padding = square_crop_box_from_points(rep_face_points_pose, rep_rgba.shape, scale=FACE_CROP_SCALE)
    rep_face_crop_rgba = crop_square_with_padding(rep_rgba, raw_box, clipped_box, padding)
    rep_face_crop_rgb = rep_face_crop_rgba[..., :3]
    rep_face_crop_512 = np.array(Image.fromarray(rep_face_crop_rgb).resize((512, 512), Image.Resampling.LANCZOS))
    rep_pose_points_512 = points_to_crop512(rep_face_points_pose, raw_box)

    # The original face is also inverted into AFHQ Dog space. YOLO pose is only used
    # to define the initial crop window; DragGAN handle points below come from the
    # GAN-projected face images, not directly from YOLO keypoints.
    org_crop_rgb = read_rgb(PROJECT_ROOT / cutout_meta["original"]["rgb_crop_path"])
    org_raw_box, org_clipped_box, org_padding = square_crop_box_from_points(org_face_points_pose, org_crop_rgb.shape, scale=FACE_CROP_SCALE)
    org_face_crop_rgb = crop_square_with_padding(org_crop_rgb, org_raw_box, org_clipped_box, org_padding, fill=(127, 127, 127, 0))
    org_face_crop_512 = np.array(Image.fromarray(org_face_crop_rgb).resize((512, 512), Image.Resampling.LANCZOS))
    org_pose_points_512 = points_to_crop512(org_face_points_pose, org_raw_box)

    print("\n" + "=" * 96)
    print(f"[02] True GAN inversion :: {idx}/{len(jobs)} :: {pair_name}")
    print("Replacement face crop raw box:", raw_box)
    print("Original face crop raw box:", org_raw_box)
    print("Replacement alpha source:", replacement_alpha_source)

    save_rgb(output_dir / "replacement_face_crop_512.png", rep_face_crop_512)
    save_rgb(output_dir / "replacement_pose_points_512.png", draw_points(rep_face_crop_512, rep_pose_points_512))
    save_rgb(output_dir / "original_face_crop_512.png", org_face_crop_512)
    save_rgb(output_dir / "original_pose_points_512.png", draw_points(org_face_crop_512, org_pose_points_512))

    # Each pair gets a fresh generator copy. PTI should specialize to the replacement
    # image only because this tuned generator is what notebook 03 will drag.
    G_pair = copy.deepcopy(G).to(DEVICE).eval().requires_grad_(False)
    w_plus, projected_face, history = project_face_to_wplus(G_pair, rep_face_crop_512, steps=INVERSION_STEPS, lr=INVERSION_LR)
    G_pair, pti_projected_face, pti_history = run_pti_tuning(G_pair, rep_face_crop_512, w_plus)
    rep_inversion_points_512, rep_inversion_point_diagnostics = estimate_inversion_face_points(pti_projected_face)

    # The original dog is inverted separately only to estimate target driver points in
    # the same GAN-aligned coordinate system. Its tuned generator is not reused later.
    G_org = copy.deepcopy(G).to(DEVICE).eval().requires_grad_(False)
    org_w_plus, org_projected_face, org_history = project_face_to_wplus(G_org, org_face_crop_512, steps=INVERSION_STEPS, lr=INVERSION_LR)
    G_org, org_pti_projected_face, org_pti_history = run_pti_tuning(G_org, org_face_crop_512, org_w_plus)
    org_inversion_points_512, org_inversion_point_diagnostics = estimate_inversion_face_points(org_pti_projected_face)

    shared_face_names = sorted(set(rep_inversion_points_512).intersection(org_inversion_points_512))
    print("Shared inversion-derived face handles:", shared_face_names)
    print("Replacement inversion point diagnostics:", rep_inversion_point_diagnostics)
    print("Original inversion point diagnostics:", org_inversion_point_diagnostics)

    tuned_generator_path = output_dir / "stylegan2-afhqdog-pti.pkl"
    if ENABLE_PTI and PTI_STEPS > 0:
        generator_pkl_for_draggan = save_generator_for_draggan(G_pair, tuned_generator_path)
    else:
        generator_pkl_for_draggan = AFHQ_DOG_PKL

    np.savez_compressed(output_dir / "projected_w_plus.npz", w=w_plus)
    np.savez_compressed(output_dir / "original_projected_w_plus.npz", w=org_w_plus)
    save_rgb(output_dir / "projected_face_512.png", projected_face)
    save_rgb(output_dir / "pti_projected_face_512.png", pti_projected_face)
    save_rgb(output_dir / "replacement_inversion_points_512.png", draw_points(pti_projected_face, rep_inversion_points_512))
    save_rgb(output_dir / "original_projected_face_512.png", org_projected_face)
    save_rgb(output_dir / "original_pti_projected_face_512.png", org_pti_projected_face)
    save_rgb(output_dir / "original_inversion_points_512.png", draw_points(org_pti_projected_face, org_inversion_points_512))

    inversion_meta = {
        "pair_name": pair_name,
        "cutout_metadata_path": str(metadata_path),
        "afhq_dog_pkl": str(AFHQ_DOG_PKL),
        "generator_pkl_for_draggan": str(generator_pkl_for_draggan),
        "replacement_alpha_source": replacement_alpha_source,
        "face_crop_raw_box_xyxy": [int(v) for v in raw_box],
        "face_crop_clipped_box_xyxy": [int(v) for v in clipped_box],
        "face_crop_padding_ltrb": [int(v) for v in padding],
        "original_face_crop_raw_box_xyxy": [int(v) for v in org_raw_box],
        "original_face_crop_clipped_box_xyxy": [int(v) for v in org_clipped_box],
        "original_face_crop_padding_ltrb": [int(v) for v in org_padding],
        "face_point_source": "gan_inversion_projection",
        "replacement_pose_face_points_512": rep_pose_points_512,
        "original_pose_face_points_512": org_pose_points_512,
        "replacement_inversion_face_points_512": rep_inversion_points_512,
        "original_inversion_face_points_512": org_inversion_points_512,
        "replacement_inversion_point_diagnostics": rep_inversion_point_diagnostics,
        "original_inversion_point_diagnostics": org_inversion_point_diagnostics,
        "replacement_face_points_512": rep_inversion_points_512,
        "original_face_points_512": org_inversion_points_512,
        "shared_face_names": shared_face_names,
        "inversion_steps": int(INVERSION_STEPS),
        "inversion_lr": float(INVERSION_LR),
        "use_lpips": bool(lpips_model is not None),
        "optimize_noise": bool(OPTIMIZE_NOISE),
        "inversion_pixel_weight": float(INVERSION_PIXEL_WEIGHT),
        "inversion_lpips_weight": float(INVERSION_LPIPS_WEIGHT),
        "inversion_w_reg_weight": float(INVERSION_W_REG_WEIGHT),
        "noise_reg_weight": float(NOISE_REG_WEIGHT),
        "enable_pti": bool(ENABLE_PTI),
        "pti_steps": int(PTI_STEPS),
        "pti_lr": float(PTI_LR),
        "pti_pixel_weight": float(PTI_PIXEL_WEIGHT),
        "pti_lpips_weight": float(PTI_LPIPS_WEIGHT),
        "history": history,
        "pti_history": pti_history,
        "original_history": org_history,
        "original_pti_history": org_pti_history,
        "outputs": {
            "face_crop_512": str(output_dir / "replacement_face_crop_512.png"),
            "projected_face_512": str(output_dir / "projected_face_512.png"),
            "pti_projected_face_512": str(output_dir / "pti_projected_face_512.png"),
            "replacement_inversion_points_512": str(output_dir / "replacement_inversion_points_512.png"),
            "original_face_crop_512": str(output_dir / "original_face_crop_512.png"),
            "original_projected_face_512": str(output_dir / "original_projected_face_512.png"),
            "original_pti_projected_face_512": str(output_dir / "original_pti_projected_face_512.png"),
            "original_inversion_points_512": str(output_dir / "original_inversion_points_512.png"),
            "w_plus": str(output_dir / "projected_w_plus.npz"),
            "original_w_plus": str(output_dir / "original_projected_w_plus.npz"),
            "generator_pkl_for_draggan": str(generator_pkl_for_draggan),
        },
    }
    with open(output_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(inversion_meta, f, ensure_ascii=False, indent=2)

    show_images([
        ("Replacement face crop", rep_face_crop_512),
        ("Replacement pose points", draw_points(rep_face_crop_512, rep_pose_points_512)),
        ("Replacement PTI projection", pti_projected_face),
        ("Replacement inversion points", draw_points(pti_projected_face, rep_inversion_points_512)),
        ("Original face crop", org_face_crop_512),
        ("Original pose points", draw_points(org_face_crop_512, org_pose_points_512)),
        ("Original PTI projection", org_pti_projected_face),
        ("Original inversion points", draw_points(org_pti_projected_face, org_inversion_points_512)),
    ], cols=4, figsize=(18, 9))

    manifest.append({
        "pair_name": pair_name,
        "metadata_path": str(output_dir / "metadata.json"),
        "output_dir": str(output_dir),
    })

with open(OUTPUT_ROOT / "batch_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Saved inversion manifest:", OUTPUT_ROOT / "batch_manifest.json")
